# RNN(순환신경망) 26.9.9
- 오래된 기억은 남아있지않음.
- 기울기 소실문제 생김.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

In [3]:
class SimpleModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleModel, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self,x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x
    


In [4]:
# 하이퍼파라미터설정

input_size = 10
hidden_size = 20
output_size = 1
batch_size = 32
num_epochs = 5
learning_rate = 0.01
max_norm = 1.0

In [5]:
# 임의의 데이터 생성하기
num_samples = 1000

X = torch.randn(num_samples, input_size)
y = torch.sum(X[:, :5], dim = 1, keepdim = True) + torch.randn(num_samples, 1) * 0.1


In [6]:
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

model = SimpleModel(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = learning_rate)


In [7]:
for epoch in range(num_epochs):
    total_loss = 0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm = max_norm)
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(epoch + 1, avg_loss)
print("학습완료")

with torch.no_grad():
    test_X = torch.randn(5, input_size)
    predictions = model(test_X)
    for i in range(5):
        print(test_X[i][:3], predictions[i].item())

1 5.264962576329708
2 4.758541516959667
3 4.295970134437084
4 3.712162137031555
5 3.094138152897358
학습완료
tensor([-1.0189, -1.2552, -0.0476]) -1.3765718936920166
tensor([-0.2409, -1.2111, -1.9678]) -0.44805672764778137
tensor([-1.4084,  0.1394, -0.3837]) -0.8873622417449951
tensor([-0.5443,  1.9778,  0.5783]) 0.5796259641647339
tensor([-0.0625, -0.0050,  0.9904]) 0.46254563331604004


In [8]:
hidden_size = 20
num_layers = 1

lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first = True) # 장단기 기억

x = torch.randn(32, 10, input_size)
output, (h_n, c_n) = lstm(x)

print(output.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([32, 10, 20])
torch.Size([1, 32, 20])
torch.Size([1, 32, 20])


In [14]:
# 한글 폰트 설정 (윈도우)
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 맑은 고딕
plt.rcParams["axes.unicode_minus"] = False       # 마이너스 깨짐 방지




if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(torch.__version__)
print(device)

2.11.0+cu128
cuda


In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers = 1):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True
        )
        self.fc = nn.Linear(hidden_size, output_size) #출력층

    def forward(self, x, h_0 = None):
        if h_0 is None:
            h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device = x.device)
        out, h_n = self.rnn(x, h_0)
        output = self.fc(out)
        return output, h_n

In [11]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

data = [torch.rand(5, 10), torch.randn(3, 10), torch.randn(7,10)]
lengths = [5, 3, 7]

padded_seq = torch.nn.utils.rnn.pad_sequence(data, batch_first = True)
print(padded_seq.shape)

packed_seq = pack_padded_sequence(padded_seq, lengths, batch_first = True, enforce_sorted = False)

print(packed_seq)

input_size = 10
hidden_size = 20
rnn = torch.nn.RNN(input_size, hidden_size, batch_first = True)

output, hidden = rnn(packed_seq)
print(output)

unpacked_output, unpacked_lengths = pad_packed_sequence(output, batch_first = True)
print(unpacked_output.shape)
print(unpacked_lengths)


torch.Size([3, 7, 10])
PackedSequence(data=tensor([[-2.5095e+00,  4.8800e-01,  7.8459e-01,  2.8647e-02,  6.4076e-01,
          5.8325e-01,  1.0669e+00, -4.5015e-01, -1.8527e-01,  7.5276e-01],
        [ 8.8227e-01,  9.1500e-01,  3.8286e-01,  9.5931e-01,  3.9045e-01,
          6.0090e-01,  2.5657e-01,  7.9364e-01,  9.4077e-01,  1.3319e-01],
        [ 7.8024e-02,  5.2581e-01, -4.8799e-01,  1.1914e+00, -8.1401e-01,
         -7.3599e-01, -8.3712e-01, -9.2239e-01, -6.3477e-02,  6.7561e-01],
        [ 4.0476e-01,  1.7847e-01,  2.6491e-01,  1.2732e+00, -1.3109e-03,
         -3.0360e-01, -1.4570e+00, -1.0234e-01, -5.9915e-01,  4.7706e-01],
        [ 9.3460e-01,  5.9358e-01,  8.6940e-01,  5.6772e-01,  7.4109e-01,
          4.2940e-01,  8.8544e-01,  5.7390e-01,  2.6658e-01,  6.2745e-01],
        [-9.7807e-02,  1.8446e+00, -1.1845e+00,  1.3835e+00,  1.0868e-02,
         -3.3874e-01, -1.3407e+00, -5.8537e-01,  5.3619e-01,  5.2462e-01],
        [ 7.2618e-01,  9.1152e-02, -3.8907e-01,  5.2792e-01, -1

In [12]:
class LSTModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers = 1, dropout = 0):
        super(LSTModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout = dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden = None):
        out, hidden = self.lstm(x, hidden)
        if isinstance(out, torch.nn.utils.rnn.PackedSequence):
            out, _ = torch.nn.utils.rnn.pad_packed_sequence(out, batch_first = True)
        out = self.dropout(out)
        output = self.fc(out)
        return output, hidden        

In [13]:
# 시계열 윈도우 만들기(슬라이딩 윈도우)
# 주식데이터로 예측한다 (1~20일데이터가지고 21일 데이터 예측함)

from sklearn.preprocessing import MinMaxScaler

if "data" in globals() and isinstance(data, list) and all(isinstance(item, torch.Tensor) for item in data):
    padded_data = torch.nn.utils.rnn.pad_sequence(data, batch_first = True)
    data_concatenated = padded_data.view(-1, padded_data.shape[-1]).numpy()
elif "data" in globals() and isinstance(data, np.ndarray):
    data_concatenated = data
else:
    print("Warning")
    data_concatenated = np.random.randn(100, 10)

scaler = MinMaxScaler()
data_normalized = scaler.fit_transform(data_concatenated)

def create_window(data, window_size, horizon = 1):
    X, y = [], []
    for i in range(len(data) - window_size - horizon +1):
        X.append(data[i:(i+window_size)])
        y.append(data[i + window_size + horizon - 1])
    return np.array(X), np.array(y)

window_size = 10
horizon = 1

X, y = create_window(data_normalized, window_size, horizon)

#데이터 분할

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]


In [15]:
# 주식가격 예측 모델 만들기

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import yfinance as yf

torch.manual_seed(42)
np.random.seed(42)

In [20]:
ticker = '005930.KS'
start_date = '2018-01-01'
end_date = '2023-01-01'
stock_data = yf.download(ticker, start = start_date, end = end_date)

df = stock_data[['Close', 'High', 'Low', 'Open', 'Volume']]
# df = stock_data
df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,005930.KS,005930.KS,005930.KS,005930.KS,005930.KS
Date,,,,,
2018-01-02,41160.609375,41467.176046,40966.988320,41451.040958,8474250
2018-01-03,41644.656250,42403.005279,41483.305393,42386.870193,10013500
2018-01-04,41209.023438,42096.453465,40854.051427,42048.048190,11695450
2018-01-05,42048.035156,42048.035156,41305.821182,41386.496614,9481150
2018-01-08,41967.359375,42370.736532,41547.847132,42273.926014,8383650


In [21]:
df = df.dropna()
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df), columns = df.columns, index = df.index
)

In [22]:
def create_sequence(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data.iloc[i:(i + seq_length)].values
        y = data.iloc[i + seq_length]['Close']
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys).reshape(-1, 1)

seq_length = 20

X, y = create_sequence(df_scaled, seq_length)
print(X.shape, y.shape)

# 시간상 순서가 중요해서 랜덤으로 짜르면 안됨.
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size + val_size], y[train_size:train_size + val_size]
X_test, y_test = X[train_size + val_size:], y[train_size + val_size:]

X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

(1210, 20, 5) (1210, 1)


In [ ]:
# 주식 예측 모델 만들기

class